# ParagAI: Train on Free Google Colab GPU (NVIDIA T4)

This notebook trains your custom **ParagAI** model from scratch on a free Google Colab GPU (NVIDIA T4).

### Step 0: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Clone the repository and enter the directory
!git clone https://github.com/paragmm/ParagAI.git
%cd /content/ParagAI

In [ ]:
# 2. Install dependencies & register package in editable mode
!pip install -r requirements.txt
!pip install -e .

In [ ]:
# 3. Verify NVIDIA CUDA GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("WARNING: Running on CPU. Switch runtime type to T4 GPU for 10x faster training!")

In [ ]:
# 4. Prepare & merge categorized datasets into combined_corpus.txt
!python scripts/prepare_data.py

In [ ]:
# 5. Run Training on GPU (CUDA acceleration)
!python scripts/train.py --data data/processed/combined_corpus.txt --config configs/train_config.yaml

In [ ]:
# 6. Test Interactive Generation in Python
import torch
from paragai.config import ModelConfig
from paragai.model.transformer import ParagAIModel
from paragai.tokenizer.char_tokenizer import CharTokenizer
from paragai.inference.generate import Generator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device)
model_config = ModelConfig(**checkpoint["model_config"])
model = ParagAIModel(model_config)
model.load_state_dict(checkpoint["model_state_dict"])
tokenizer = CharTokenizer.load("checkpoints/tokenizer.json")

generator = Generator(model, tokenizer, device=device)

# Prompt the trained model across categories
prompt = "[CATEGORY: stories] একদা"
output = generator.generate(prompt, max_new_tokens=100, temperature=0.7)
print("\n--- Generated Story ---")
print(output)

prompt_tech = "[CATEGORY: technology] Python is"
output_tech = generator.generate(prompt_tech, max_new_tokens=100, temperature=0.7)
print("\n--- Generated Tech ---")
print(output_tech)

In [ ]:
# 7. Export to GGUF format for Ollama (Optional)
!python scripts/export_hf.py --checkpoint checkpoints/best_model.pt --output_dir checkpoints/export_hf
!python scripts/export_gguf.py --input_dir checkpoints/export_hf --tokenizer checkpoints/tokenizer.json --output checkpoints/paragai.gguf

In [ ]:
# 8. Download trained model checkpoint & GGUF to your PC
from google.colab import files
files.download("checkpoints/best_model.pt")
files.download("checkpoints/tokenizer.json")
# If exported to GGUF:
# files.download("checkpoints/paragai.gguf")